# Car Price Prediction Project Modeling
This project aims to build and evaluate machine learning models for predicting used car prices. The pipeline is structured into the following key steps:

1.  **Advanced Data Cleaning**: Initial processing of raw car data to remove outliers, handle missing values, and prepare the target variable ('price') through logarithmic transformation for better model performance. We also compute a 'vehicle age' feature.
2.  **Indicator Engineering**: Creation of several categorical indicator features based on vehicle characteristics and listing titles (e.g., luxury brands, power features, utility keywords) to enrich the dataset.
3.  **Feature Pipeline**: Transformation of raw and engineered features into a format suitable for machine learning models. This involves Target Encoding for high-cardinality categorical features, One-Hot Encoding for low-cardinality features, and handling numerical data. All features are then stacked into sparse matrices.
4.  **Model Comparison**: Definition and instantiation of various regression models, including linear models (Lasso, Ridge), tree-based models (Decision Tree, Random Forest), and gradient boosting machines (XGBoost, CatBoost).
5.  **Execution & Results**: Training and evaluation of all defined models using the prepared dataset. Performance is measured using Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and Mean Absolute Percentage Error (MAPE).

The primary objective of this project is to minimize the prediction error, specifically targeting the **Root Mean Squared Error (RMSE)**. A lower RMSE indicates a more accurate model in predicting car prices. The secondary objective is to evaluate the **Mean Absolute Percentage Error (MAPE)** to understand the average percentage deviation of predictions from actual prices, providing a more intuitive measure of accuracy from a business perspective. We aim to identify the model that achieves the lowest RMSE and MAPE on the test set while maintaining reasonable training and prediction times.

#0.0 Install Necessary Libraries if not already installed

In [ ]:
!pip install catboost
!pip install xgboost
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.7 MB/s eta 0:00:00


#0.5 Importing necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import os
import time
import warnings
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')

print("Libraries installed successfully!")

Libraries installed successfully!


# 1. Advanced Data Cleaning


This section loads the car dataset, performs initial data cleaning by removing misleading or out-of-range price values, calculates the vehicle's 'age' based on the model year, applies a logarithmic transformation to the 'price' target variable for better model performance, and cleans the listing titles for further feature engineering.

In [ ]:
print("Loading and cleaning data...")

# Importing Data
file_name = "cleaned_car_data.csv"
found_path = None

import os
import pandas as pd

search_path = "/content"
file_name = "cleaned_car_data.csv"

found_path = None

for root, dirs, files in os.walk(search_path):
    if file_name in files:
        found_path = os.path.join(root, file_name)
        break

if found_path:
    print(f"File '{file_name}' found at: {found_path}")
    df = pd.read_csv(found_path)
else:
    raise FileNotFoundError(
        f"File '{file_name}' not found in '{search_path}' or its subdirectories."
    )


# Cleaning Missleading Values that would potentially skew our resuçts
df = df[~df["price"].isin([123456, 111111, 1, 0])]
df = df[(df["price"] >= 1500) & (df["price"] <= 90000)]

# FEATURE: Vehicle Age (Depreciation)
if 'vehicle_model_year' in df.columns:
    df['age'] = 2026 - df['vehicle_model_year']
    df = df[(df['age'] >= 0) & (df['age'] <= 25)] # Remove vintage/invalid data

# Target Log Transform
df["log_price"] = np.log1p(df["price"])

# Title Cleaning
df["title_clean"] = df["marketplace_listing_title"].astype(str).str.lower().str.replace(r"[^a-z0-9\s]", " ", regex=True)

print("Data cleaning completed successfully!")

Loading and cleaning data...


FileNotFoundError: File 'cleaned_car_data.csv' not found in '/content' or its subdirectories.

# 2. Indicator Engineering

Here, the `apply_indicators` function is defined and used to create several boolean indicator features. These features capture aspects such as whether a vehicle belongs to a luxury brand (`is_lux_brand`), if its title contains luxury-related keywords (`lux_score`), power-related keywords (`pwr_score`), or utility/comfort-related keywords (`util_score`).

In [ ]:
def apply_indicators(df):
    lux_brands = ['mercedes-benz', 'bmw', 'audi', 'lexus', 'porsche', 'land rover', 'tesla']
    lux_kws = ['leather', 'sunroof', 'premium', 'platinum', 'luxury', 'limited']
    df['is_lux_brand'] = df['vehicle_make_display_name'].str.lower().isin(lux_brands).astype(int)
    df['lux_score'] = df['title_clean'].str.contains('|'.join(lux_kws)).astype(int)

    pwr_kws = ['v8', 'v6', 'turbo', 'sport', '4wd', 'awd', 'hemi', 'supercharged']
    df['pwr_score'] = df['title_clean'].str.contains('|'.join(pwr_kws)).astype(int)

    util_kws = ['heated', 'captain', 'spacious', 'third row', 'towing', 'navigation']
    df['util_score'] = df['title_clean'].str.contains('|'.join(util_kws)).astype(int)
    return df

df = apply_indicators(df)

print("Indicators applied successfully!")
df.head(10)


# 3. Feature Pipeline

This section sets up the feature engineering pipeline. It categorizes columns for Target Encoding (high cardinality) and One-Hot Encoding (low cardinality), splits the dataset into training and testing sets, applies the respective encoders, processes numerical features, and finally stacks all processed features into sparse matrices (`X_train_final`, `X_test_final`) ready for model training.

In [ ]:
# In columns with high cardinality is better to apply Target Encoding (Model + State)
te_cols = ["vehicle_model_display_name", "state"]
# Low cardinality -> One Hot
ohe_cols = ["vehicle_fuel_type", "vehicle_transmission_type"]

# Identify numeric columns (including Age and Scores)
exclude = te_cols + ohe_cols + ["price", "log_price", "marketplace_listing_title", "title_clean", "vehicle_model_year", "vehicle_make_display_name"]
numeric_cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['int64', 'float64', 'int32']]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(df, df["log_price"], test_size=0.2, random_state=42)

# --- Encoding ---
te = TargetEncoder(target_type="continuous", smooth="auto", random_state=42)
X_te_train = te.fit_transform(X_train_raw[te_cols], y_train)
X_te_test  = te.transform(X_test_raw[te_cols])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_ohe_train = ohe.fit_transform(X_train_raw[ohe_cols])
X_ohe_test  = ohe.transform(X_test_raw[ohe_cols])

X_num_train = csr_matrix(X_train_raw[numeric_cols].values)
X_num_test  = csr_matrix(X_test_raw[numeric_cols].values)

# Stack
X_train_final = hstack([X_num_train, X_te_train, X_ohe_train])
X_test_final  = hstack([X_num_test,  X_te_test,  X_ohe_test])

print("Feature pipeline completed successfully!")

In [ ]:
X_test_final

NameError: name 'X_test_final' is not defined

# 4. Model Comparison

This block defines a dictionary named `models` containing various regression algorithms that will be used for comparative benchmarking. Each model is instantiated with specific hyperparameters, including linear models (Lasso, Ridge), tree-based models (Decision Tree, Random Forest), and gradient boosting machines (XGBoost, CatBoost), with early stopping configured for the boosting models.

In [ ]:
models = {
    "Lasso (L1)": Lasso(alpha=0.01),
    "Ridge (L2)": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, n_jobs=-1, random_state=42, early_stopping_rounds=50),
    "CatBoost": CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, verbose=0, random_seed=42, early_stopping_rounds=50)
}

print("Models defined successfully!")


# 5. Execution & Results

In this section, a comparative benchmark is performed across all defined models. Each model is trained on the prepared data, and predictions are made on the test set. Performance metrics, including Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and Mean Absolute Percentage Error (MAPE), are calculated for the real (untransformed) price values. Finally, a leaderboard table summarizing the performance of all models is printed.

In [ ]:
results = []
print("\n--- Starting Comparative Benchmark ---")

for name, model in models.items():
    start = time.time()

    if name in ["XGBoost", "CatBoost"]:
        # early_stopping_rounds moved to model constructor
        model.fit(X_train_final, y_train, eval_set=[(X_test_final, y_test)], verbose=False)
    else:
        model.fit(X_train_final, y_train)

    # Predictions
    p_log = model.predict(X_test_final)
    p_real = np.expm1(p_log)
    a_real = np.expm1(y_test.values)

    rmse = np.sqrt(mean_squared_error(a_real, p_real))
    mae = mean_absolute_error(a_real, p_real)
    mape = np.mean(np.abs((a_real - p_real) / a_real)) * 100

    print(f"Finished {name:15} | RMSE: ${rmse:,.0f} | MAPE: {mape:.1f}%")
    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "MAPE %": mape})

# Final Table
results_df = pd.DataFrame(results).sort_values("RMSE")
print("\n=== FINAL LEADERBOARD ===")
print(results_df.to_string(index=False))

In [ ]:
xgb_model = models["XGBoost"]

# Reconstruct feature names in the same order as hstack
feature_names = (
    numeric_cols +  # Numeric columns
    te_cols +  # Target encoded columns
    ohe.get_feature_names_out(ohe_cols).tolist()  # One-hot encoded columns
)

# Get feature importances
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n=== XGBoost Feature Importances ===")
print(feature_importance.to_string(index=False))